# Common Task II: GNN Classification
Graph representation preserves spatial relationships between particles.
GNN improves classification by modeling local interactions.

In [1]:
import os; os.chdir('/Users/sasisundar/Desktop/ml4sci-gsoc/ml4sci-gsoc')
import sys; sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')


In [2]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from torch.utils.data import Subset
from src.models.gnn import GNN
from src.data.graph_loader import GraphJetDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Use the SAME dataset and split as training
dataset = GraphJetDataset('dataset.hdf5', max_samples=5000)
labels = dataset.get_labels()
indices = list(range(len(dataset)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)
test_dataset = Subset(dataset, test_idx)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = GNN().to(device)
model.load_state_dict(torch.load('outputs/models/gnn.pt', map_location=device, weights_only=True))
model.eval()

all_preds, all_labels, all_probs = [], [], []
print('Evaluating on held-out test set...')
for batch in test_loader:
    batch = batch.to(device)
    out = model(batch)
    probs = torch.softmax(out, dim=1)[:,1]
    preds = torch.argmax(out, dim=1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch.y.cpu().numpy())
    all_probs.extend(probs.detach().cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
print(f'Test Accuracy: {acc:.4f}')
print(f'Test ROC AUC: {auc:.4f}')


Device: cpu


Evaluating on held-out test set...


Test Accuracy: 0.6930
Test ROC AUC: 0.7833
